# C3 — Dự báo Hoạt động Đại lý (Churn Prediction)
**DATA EXPLORERS 2026 | Xe đạp Thống Nhất**

Dự báo đại lý nào có nguy cơ ngừng mua hàng (churn) trong Q2/2026:
- Xây dựng đặc trưng **RFM** (Recency, Frequency, Monetary)
- Huấn luyện **XGBoost Classifier**
- Phân tích **feature importance**
- Xuất danh sách đại lý rủi ro cao

**Thứ tự chạy:** tuần tự từ trên xuống (Run All)


## 1. Thư viện & Cài đặt

In [ ]:
# thư viện
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings('ignore')

from xgboost import XGBClassifier
from sklearn.model_selection import train_test_split, cross_val_score, StratifiedKFold
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import (roc_auc_score, classification_report,
                              confusion_matrix, RocCurveDisplay)

## 2. Tải & Làm sạch Dữ liệu

In [ ]:
# tải dữ liệu giao dịch đầy đủ
df_fact = pd.read_csv('data/fact_sales_full.csv', parse_dates=['order_date'])
print(f'Shape: {df_fact.shape}')
print(f'Khoảng thời gian: {df_fact.order_date.min().strftime("%m/%Y")} – {df_fact.order_date.max().strftime("%m/%Y")}')
df_fact.head()

In [ ]:
# kiểm tra missing
print('Missing values:')
print(df_fact.isnull().sum()[df_fact.isnull().sum() > 0])

# thống kê đại lý
print(f'\nSố đại lý: {df_fact.customer_code.nunique():,}')
print(f'Số đơn hàng: {len(df_fact):,}')
print(f'Tổng doanh thu: {df_fact.line_total.sum()/1e9:.1f} tỷ VND')

In [ ]:
# làm sạch: bỏ dòng thiếu thông tin đại lý
df_clean = df_fact.dropna(subset=['customer_code', 'customer_name']).copy()
df_clean['line_total'] = df_clean['line_total'].fillna(0)
df_clean['quantity']   = df_clean['quantity'].fillna(0)
print(f'Sau clean: {len(df_clean):,} đơn ({len(df_clean)/len(df_fact)*100:.1f}%)')

## 3. Xây dựng Đặc trưng RFM

In [ ]:
# ngày tham chiếu = ngày cuối dữ liệu
reference_date = df_clean['order_date'].max()
print(f'Ngày tham chiếu: {reference_date.strftime("%d/%m/%Y")}')

# RFM cơ bản
rfm = df_clean.groupby('customer_code').agg(
    recency   = ('order_date', lambda x: (reference_date - x.max()).days),
    frequency = ('so_number',  'nunique'),
    monetary  = ('line_total', 'sum')
).reset_index()

# đặc trưng bổ sung: đơn hàng 3 tháng gần nhất
cutoff_3m = reference_date - pd.DateOffset(months=3)
last3m = (df_clean[df_clean.order_date >= cutoff_3m]
          .groupby('customer_code')['so_number']
          .nunique().rename('last3m_orders'))
rfm = rfm.merge(last3m, on='customer_code', how='left').fillna({'last3m_orders': 0})

print(f'\nRFM shape: {rfm.shape}')
print(rfm.describe().round(0))

In [ ]:
# phân phối RFM
fig, axes = plt.subplots(1, 3, figsize=(15, 5))
for ax, col, title in zip(axes,
        ['recency', 'frequency', 'monetary'],
        ['Recency (ngày)', 'Frequency (đơn)', 'Monetary (VND)']):
    ax.hist(rfm[col], bins=25, alpha=0.8, edgecolor='white')
    ax.axvline(rfm[col].median(), linestyle='--', linewidth=1.5, label=f'Trung vị: {rfm[col].median():.0f}')
    ax.set_title(title, fontsize=11, fontweight='bold')
    ax.set_ylabel('Số đại lý')
    ax.legend(fontsize=9)
    ax.grid(True, alpha=0.3)
plt.suptitle('Phân phối RFM — Đại lý Thống Nhất Bike', fontsize=13, fontweight='bold')
fig.tight_layout()
plt.show()

## 4. Gán nhãn Churn

In [ ]:
# định nghĩa churn: recency > 90 ngày (không mua trong 3 tháng)
CHURN_THRESHOLD = 90
rfm['churn'] = (rfm['recency'] > CHURN_THRESHOLD).astype(int)

churn_rate = rfm['churn'].mean()
print(f'Ngưỡng churn: recency > {CHURN_THRESHOLD} ngày')
print(f'Số đại lý churn: {rfm.churn.sum()} / {len(rfm)} ({churn_rate*100:.1f}%)')
print(f'Số đại lý active: {(rfm.churn==0).sum()} ({(1-churn_rate)*100:.1f}%)')

In [ ]:
# phân bố churn theo recency — scatter
fig, ax = plt.subplots(figsize=(12, 5))
churn0 = rfm[rfm.churn == 0]
churn1 = rfm[rfm.churn == 1]

ax.scatter(churn0.recency, churn0.monetary/1e6, alpha=0.5, s=30, label='Active')
ax.scatter(churn1.recency, churn1.monetary/1e6, alpha=0.5, s=30, label='Churn', marker='x')
ax.axvline(CHURN_THRESHOLD, linestyle='--', alpha=0.7, label=f'Ngưỡng {CHURN_THRESHOLD} ngày')
ax.set_xlabel('Recency (ngày)')
ax.set_ylabel('Monetary (triệu VND)')
ax.set_title('Phân bố Đại lý: Active vs Churn', fontsize=13, fontweight='bold')
ax.legend(fontsize=10)
ax.grid(True, alpha=0.3)
fig.tight_layout()
plt.show()

## 5. Mô hình XGBoost Classifier

In [ ]:
# lấy tham số tốt nhất từ Parameter Tuning
params_clf = pd.read_csv('best_params/best_params_classifier.csv', index_col=0)
n_estimators = int(params_clf.loc['n_estimators'][0])
max_depth    = int(params_clf.loc['max_depth'][0])
lr           = float(params_clf.loc['learning_rate'][0])
subsample    = float(params_clf.loc['subsample'][0])
print('Tham số XGBoost:')
params_clf

In [ ]:
# chuẩn bị dữ liệu
feature_cols = ['recency', 'frequency', 'monetary', 'last3m_orders']
X = rfm[feature_cols].values
y = rfm['churn'].values

# scale
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

# tách train/test
X_train, X_test, y_train, y_test = train_test_split(
    X_scaled, y, test_size=0.2, random_state=42, stratify=y
)
print(f'Train: {len(X_train)} | Test: {len(X_test)}')
print(f'Churn rate train: {y_train.mean()*100:.1f}% | test: {y_test.mean()*100:.1f}%')

In [ ]:
# huấn luyện XGBoost
clf = XGBClassifier(
    n_estimators  = n_estimators,
    max_depth     = max_depth,
    learning_rate = lr,
    subsample     = subsample,
    use_label_encoder=False,
    eval_metric   = 'logloss',
    random_state  = 42
)
clf.fit(X_train, y_train)
print('✅ XGBoost đã huấn luyện xong')

## 6. Đánh giá Mô hình

In [ ]:
# cross-validation AUC
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
auc_scores = cross_val_score(clf, X_scaled, y, cv=cv, scoring='roc_auc')
print(f'AUC-ROC (5-fold CV):')
print(f'  Mean : {auc_scores.mean():.4f}')
print(f'  Std  : {auc_scores.std():.4f}')
print(f'  Scores: {np.round(auc_scores, 4)}')

In [ ]:
# báo cáo phân loại trên test set
y_pred = clf.predict(X_test)
y_prob = clf.predict_proba(X_test)[:, 1]

print(f'AUC-ROC (test): {roc_auc_score(y_test, y_prob):.4f}')
print()
print(classification_report(y_test, y_pred, target_names=['Active','Churn']))

In [ ]:
# confusion matrix + ROC curve
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

# confusion matrix
cm = confusion_matrix(y_test, y_pred)
im = ax1.imshow(cm, interpolation='nearest', aspect='auto')
ax1.set_title('Confusion Matrix', fontsize=12, fontweight='bold')
for i in range(2):
    for j in range(2):
        ax1.text(j, i, str(cm[i, j]), ha='center', va='center', fontsize=14, fontweight='bold')
ax1.set_xticks([0, 1]); ax1.set_yticks([0, 1])
ax1.set_xticklabels(['Active','Churn']); ax1.set_yticklabels(['Active','Churn'])
ax1.set_xlabel('Dự báo'); ax1.set_ylabel('Thực tế')

# ROC curve
RocCurveDisplay.from_predictions(y_test, y_prob, ax=ax2)
ax2.plot([0,1],[0,1],'--', alpha=0.5, label='Random')
ax2.set_title('ROC Curve', fontsize=12, fontweight='bold')
ax2.legend(fontsize=9)
ax2.grid(True, alpha=0.3)

fig.tight_layout()
plt.show()

## 7. Tầm quan trọng Đặc trưng

In [ ]:
# feature importance
fi = pd.Series(clf.feature_importances_, index=feature_cols).sort_values(ascending=True)

fig, ax = plt.subplots(figsize=(10, 5))
ax.barh(fi.index, fi.values, alpha=0.85)
for i, v in enumerate(fi.values):
    ax.text(v + 0.005, i, f'{v:.3f}', va='center', fontsize=10, fontweight='bold')
ax.set_xlabel('Importance Score')
ax.set_title('Feature Importance — XGBoost Churn Model', fontsize=13, fontweight='bold')
ax.grid(True, alpha=0.3, axis='x')
fig.tight_layout()
plt.show()

## 8. Danh sách Đại lý Rủi ro Cao

In [ ]:
# dự báo xác suất churn trên toàn bộ đại lý
rfm['churn_prob'] = clf.predict_proba(X_scaled)[:, 1]
rfm['churn_pred'] = clf.predict(X_scaled)
rfm['risk_level'] = pd.cut(rfm['churn_prob'],
                             bins=[0, 0.3, 0.6, 1.0],
                             labels=['Thấp', 'Trung bình', 'Cao'])

# gắn tên đại lý
name_map = df_clean.drop_duplicates('customer_code')[['customer_code','customer_name']]
rfm_named = rfm.merge(name_map, on='customer_code', how='left')

# top 20 nguy cơ cao nhất
high_risk = (rfm_named[rfm_named.risk_level == 'Cao']
             .sort_values('churn_prob', ascending=False)
             .head(20))

print(f'Phân bố mức độ rủi ro:')
print(rfm.risk_level.value_counts().to_string())
print(f'\nTop 20 đại lý rủi ro cao nhất:')
high_risk[['customer_name','recency','frequency','monetary','churn_prob']].to_string(index=False)

In [ ]:
# biểu đồ đại lý rủi ro cao
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(16, 6))

# phân bố xác suất churn
ax1.hist(rfm['churn_prob'], bins=25, alpha=0.8, edgecolor='white')
ax1.axvline(0.5, linestyle='--', linewidth=2, label='Ngưỡng 0.5')
ax1.axvline(0.6, linestyle=':', linewidth=2, label='Rủi ro cao 0.6')
ax1.set_xlabel('Xác suất Churn')
ax1.set_ylabel('Số đại lý')
ax1.set_title('Phân bố Xác suất Churn', fontsize=12, fontweight='bold')
ax1.legend()
ax1.grid(True, alpha=0.3)

# top 15 rủi ro cao nhất
top15 = high_risk.head(15)
ax2.barh(range(len(top15)), top15['churn_prob'], alpha=0.85)
ax2.set_yticks(range(len(top15)))
ax2.set_yticklabels([n[:35] if pd.notna(n) else 'N/A'
                     for n in top15['customer_name']], fontsize=8)
ax2.axvline(0.5, linestyle='--', alpha=0.7)
ax2.set_xlabel('Xác suất Churn')
ax2.set_title('Top 15 Đại lý Rủi ro Churn Cao nhất', fontsize=12, fontweight='bold')
ax2.grid(True, alpha=0.3, axis='x')

fig.suptitle('Kết quả Dự báo Churn — Đại lý Thống Nhất Bike', fontsize=13, fontweight='bold')
fig.tight_layout()
plt.show()

## 9. Xuất Kết quả

In [ ]:
# lưu danh sách đại lý churn
import os
os.makedirs('../Forecasting Product/Ensemble', exist_ok=True)

rfm_out = rfm_named[['customer_code','customer_name',
                       'recency','frequency','monetary',
                       'last3m_orders','churn_prob','churn_pred','risk_level']]
rfm_out.to_csv('output/predictions_gbm.csv', index=False)
print('✅ Đã lưu predictions_gbm.csv')
print(f'   Tổng: {len(rfm_out)} đại lý')
print(f'   Rủi ro Cao  : {(rfm_out.risk_level=="Cao").sum()}')
print(f'   Rủi ro TB   : {(rfm_out.risk_level=="Trung bình").sum()}')
print(f'   Rủi ro Thấp : {(rfm_out.risk_level=="Thấp").sum()}')

In [ ]:
# tóm tắt kết quả
auc_final = roc_auc_score(y_test, y_prob)
print('=' * 55)
print('   KẾT QUẢ C3 — DỰ BÁO CHURN ĐẠI LÝ')
print('=' * 55)
print(f'  Tổng đại lý phân tích : {len(rfm):,}')
print(f'  Đại lý Churn (thực tế): {rfm.churn.sum():,} ({rfm.churn.mean()*100:.1f}%)')
print(f'  AUC-ROC                : {auc_final:.4f}')
print(f'  Đặc trưng quan trọng  : {fi.idxmax()} ({fi.max():.3f})')
print(f'  Đại lý rủi ro cao     : {(rfm.risk_level=="Cao").sum()}')
print('=' * 55)